# LLM 9일차 (3) — 멀티 에이전트 RAG 챗봇(동기)의 수업 실행 기록 읽기 (`LangGraph_실습5` 완성본)

어제(Day 053) 재구성 체인에서 멈춰 있던 `LangGraph_실습5` 를 수업에서 **완성해 진짜 gpt-5-nano 로 3턴 대화**한 판이다. 코드 정리 · 키 없이 돌리는 판은 Day 053 [`langgraph_multi_agent_rag.ipynb`](../../2026-11-23/lecture/langgraph_multi_agent_rag.ipynb) 에 있고, 여기서는 **수업 출력**을 옮겨 읽었다.

> **출력 읽는 법** — `# 📼` 셀은 수업 시간 출력을 그대로 옮긴 것이다(코드는 실행하지 않음 · 키 필요).

## 1. 수업 완성본의 구조 — Day 053 정리판과 다른 점

| 부분 | 수업 완성본 (`실습5`) | Day 053 정리판 |
|---|---|---|
| 상태 | `messages` · `rephrased_query` · `context` | + `next` (교재 p.113 방식) |
| 흐름 | `add_edge` 3개 (직선) | `next` + `router` 조건부 엣지 — 결과는 같다 |
| 재구성 | **기록이 있을 때만** `rephrase_chain` 호출 (첫 질문은 그대로) | 매번 호출 |
| 답변 체인 | `qa_prompt | llm` → `AIMessage` 를 그대로 `messages` 에 | `| StrOutputParser()` 후 `AIMessage(...)` 로 다시 감쌈 |
| 대화 이어 가기 | `inputs['messages'].append(질문)` → `final_state = invoke(inputs)` → `inputs = final_state` | `chat + [질문]` 으로 넘김 |

- 수업판의 '기록이 있을 때만 재구성' 은 첫 질문에서 LLM 호출 1번을 아끼는 좋은 방법이다

## 2. 수업 실행 — 3턴 대화

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

inputs = {"messages": []}
while True:
    user_input = input("사용자: ")
    if user_input.lower() == 'exit':
        print("챗봇을 안전하게 종료합니다."); break
    inputs["messages"].append(HumanMessage(content=user_input))
    final_state = multi_agent.invoke(inputs)
    print(f"AI 에이전트:\n{final_state['messages'][-1].content}")
    print("-" * 80)
    inputs = final_state

안녕하세요! 코드크래프터스 아카데미 지원 챗봇입니다. 무엇이든 물어보세요! (종료: exit)
에이전트 1: 질문 맥락 분석 및 재구성
--------------------------------------------------------------------------------
state: 
messages=[HumanMessage(content='수강 신청한 강의를 환불하고 싶어요', additional_kwargs={}, response_metadata={}, id='9ba4c2f6-507b-465c-b1b8-0a536ba4c4f8')] rephrased_query='' context=''
--------------------------------------------------------------------------------
[결과] 재구성된 쿼리: 수강 신청한 강의를 환불하고 싶어요
에이전트 2: 관련 매뉴얼 검색
답변의 근거: 
question: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?
answer: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.
--------------------------------------------------------------------------------
에이전트 3: 최종 답변 작성
--------------------------------------------------------------------------------
AI 에이전트:
환불은 가능합니다. 조건은 아래 두 가지입니다.
- 결제일로부터 7일 이내
- 강의 수강률이 5% 미만

두 조건을 모두 충족하면 전액 환불해 드립니다. 처리은 고객센터를 통해 문의하시면 신속하게 진행됩니다.

도움이 필요하시면 강의명, 주문번호, 결제일, 현재 수강률 정보를 알려주시면 제가 조건 충족 여부를 함께 점검하

## 3. 출력에서 확인한 것

| 턴 | 사용자 | 재구성된 쿼리 | 찾은 FAQ |
|---|---|---|---|
| 1 | 수강 신청한 강의를 환불하고 싶어요 | (기록 없음 → 그대로) | 환불 정책 |
| 2 | 그거 전액 다 받으려면 조건이 어떻게 되나요? | 수강한 강의를 전액 환불받으려면 어떤 조건을 충족해야 합니까? | 환불 정책 ✅ |
| 3 | (같은 질문 한 번 더) | 수강 신청한 강의를 전액 환불받으려면 어떤 조건을 충족해야 합니까? | 환불 정책 ✅ |

- '그거' 를 **'수강한 강의를 전액 환불'** 로 풀어 쓴 것이 핵심 — Day 053 에서 🧪 '이어 붙이기' 규칙으로 흉내 낸 일을 진짜 LLM 은 한 문장으로 다시 썼다
- 답의 조건(결제일로부터 7일 이내 · 수강률 5% 미만 · 둘 다 충족 시 전액)은 FAQ-002 와 일치한다
- 📝 **상태 출력의 묵은 값** — 2턴 시작 `state` 에 `rephrased_query='수강 신청한 강의를 환불하고 싶어요'` · 1턴 `context` 가 그대로 들어 있다. `inputs = final_state` 로 **최종 상태 전체**를 다음 입력으로 넘겨서다. 이번에는 노드가 곧 덮어써 문제없지만, 재구성이 실패해 값을 안 쓰면 **앞 턴 검색어로 검색**하게 된다. `inputs = {'messages': final_state['messages']}` 처럼 대화 기록만 넘기는 편이 안전하다
- 📝 **할 수 없는 일을 약속** — 답변이 '강의명 · 주문번호 · 결제일 · 수강률을 알려주시면 제가 조건 충족 여부를 점검하고 **고객센터로 전달해 드리겠습니다**' 라고 했다. 이 챗봇에는 주문 조회 · 고객센터 전달 기능(도구)이 없고 FAQ 에도 없는 내용이다. 시스템 프롬프트에 '매뉴얼에 없는 절차 · 기능을 약속하지 말 것' 을 넣어야 한다
- **추론 토큰** — 답변 1 · 2 의 `completion_tokens` 1022 · 834 중 `reasoning_tokens` 가 **896 · 704 (88% · 84%)**. 보이는 답보다 '생각' 에 훨씬 많은 토큰을 썼다 → 비용 · 시간. 짧은 FAQ 응대라면 `reasoning_effort='minimal'` 을 시험해 볼 만하다(Day 053 메모)
- LLM 출력 맞춤법 '처리은' → '처리는' (모델이 만든 오타)